In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import RidgeClassifierCV, LogisticRegression, SGDClassifier, PassiveAggressiveClassifier
# 生成哈德玛矩阵
def hadamard_matrix(n):
    if n == 1:
        return np.array([[1]])
    elif n % 2 != 0 or (n & (n - 1) != 0):
        raise ValueError("哈德玛矩阵的阶数必须是 2 的幂！")
    else:
        H = hadamard_matrix(n // 2)
        return np.block([
            [H, H],
            [H, -H]
        ])

# HITRocketTransform 类
class HITRocketTransform(nn.Module):
    def __init__(self, dilations, bias_quantiles, num_samples=1, hadamard_order=8):
        super(HITRocketTransform, self).__init__()
        self.dilations = dilations
        self.bias_quantiles = torch.tensor(bias_quantiles, dtype=torch.float32)  # 确保是张量
        self.kernel_size = hadamard_order  # 卷积核大小，使用哈德玛矩阵的阶数
        self.num_kernels = hadamard_order  # 卷积核数量，使用哈德玛矩阵的列数
        self.num_samples = num_samples  # 用于计算偏置的样本数量
        
        # 初始化卷积核权重，使用哈德玛矩阵的列
        self.weights = self._initialize_weights(hadamard_order)
        
        # 初始化偏置
        self.biases = None

    def _initialize_weights(self, hadamard_order):
        H = hadamard_matrix(hadamard_order)
        weights = torch.tensor(H.T, dtype=torch.float32)  # 使用哈德玛矩阵的转置作为卷积核权重
        return nn.Parameter(weights.unsqueeze(1), requires_grad=False)

    def _fit_biases(self, x):
        random_sample_indices = np.random.choice(x.shape[0], self.num_samples, replace=False)
        x_samples = x[random_sample_indices]

        biases = []

        for dilation in self.dilations:
            padding = (self.kernel_size - 1) * dilation // 2
            x_padded = F.pad(x_samples, (padding, padding))
            conv_outputs = F.conv1d(x_padded, self.weights, dilation=dilation, padding=0)

            conv_outputs_flattened = conv_outputs.view(self.num_samples, self.num_kernels, -1)
            quantiles = torch.quantile(conv_outputs_flattened, self.bias_quantiles, dim=2)

            # 重塑为 (num_samples, num_kernels, len(bias_quantiles))
            quantiles = quantiles.permute(1, 2, 0).reshape(self.num_samples, self.num_kernels, len(self.bias_quantiles))

            biases.append(quantiles)

        # 将所有偏置合并为一个张量
        self.biases = torch.cat(biases, dim=1)  # 形状为 (num_samples, num_kernels * len(dilations) * len(bias_quantiles))

    def forward(self, x):
        if self.biases is None:
            self._fit_biases(x)

        batch_size, _, signal_length = x.shape
        features = []

        for dilation in self.dilations:
            padding = (self.kernel_size - 1) * dilation // 2
            x_padded = F.pad(x, (padding, padding))
            conv_output = F.conv1d(x_padded, self.weights, dilation=dilation, padding=0)

            # 对每个卷积核和膨胀因子单独计算特征
            for kernel_idx in range(self.num_kernels):
                for bias_idx in range(len(self.bias_quantiles)):
                    for sample_idx in range(self.num_samples):
                        bias = self.biases[sample_idx, kernel_idx, bias_idx]
                        ppv = (conv_output[:, kernel_idx, :] > bias).float().mean(dim=-1)
                        features.append(ppv)

        return torch.stack(features, dim=1)


In [6]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV, LogisticRegression
from sklearn.metrics import f1_score
import time

# 假设 HITRocketTransform 类已定义
# from your_module import HITRocketTransform  # 如果你是通过自定义模块引入
def _quantiles_torch(n: int) -> torch.Tensor:
    phi = (np.sqrt(5) + 1) / 2
    return torch.tensor([(i * phi) % 1 for i in range(1, n + 1)], dtype=torch.float32, device='cuda')

def HITrocket_Liear_Pro(X_train, y_train, X_test, y_test, repeat_times=1):
    X_train = np.nan_to_num(X_train, nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)




    dilations = list(range(1, 12))
    n_quantiles = 5
    bias_quantiles_tensor = _quantiles_torch(n_quantiles)
    bias_quantiles = bias_quantiles_tensor.cpu().tolist()  # 转成python list更直观

    num_samples = 5

    results_sum = {}
    total_train_time = 0
    dimension = None

    linear_models = {
        "RidgeClassifierCV": RidgeClassifierCV(
            alphas=[0.01, 0.1, 1.0, 10.0],
            class_weight='balanced',
            store_cv_values=True
        ),
        "LogisticRegression": LogisticRegression(
            C=1.0,
            penalty='l2',
            solver='liblinear',
            max_iter=1000,
            class_weight='balanced'
        ),
    }

    # 初始化模型结果容器
    for name in linear_models:
        results_sum[name] = {
            "Train Time (s)": 0.0,
            "Weighted F1": 0.0
        }

    for i in range(repeat_times):
        print(f"🔁 第 {i + 1}/{repeat_times} 次训练")

        start_time = time.time()

        all_segments_tensor_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
        labels_tensor_train = torch.tensor(y_train, dtype=torch.long)
        all_segments_tensor_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
        labels_tensor_test = torch.tensor(y_test, dtype=torch.long)

        model = HITRocketTransform(dilations, bias_quantiles, num_samples, hadamard_order=16)
        features_train = model(all_segments_tensor_train).numpy()
        features_test = model(all_segments_tensor_test).numpy()
        labels_train = labels_tensor_train.numpy()
        labels_test = labels_tensor_test.numpy()

        # 标准化
        scaler = StandardScaler(with_mean=False)
        X_train_feat = scaler.fit_transform(features_train)
        X_test_feat = scaler.transform(features_test)
        dimension = X_train_feat.shape[1]
        print(dimension)
        # 每个模型训练和预测
        for name, clf in linear_models.items():
            print(f"    📐 模型: {name}")
            t0 = time.time()
            clf.fit(X_train_feat, labels_train)
            train_time = time.time() - t0

            y_pred = clf.predict(X_test_feat)
            f1 = f1_score(labels_test, y_pred, average='weighted')

            results_sum[name]["Train Time (s)"] += train_time
            results_sum[name]["Weighted F1"] += f1

        total_train_time += time.time() - start_time

    # 取平均值
    for name in results_sum:
        results_sum[name]["Train Time (s)"] /= repeat_times
        results_sum[name]["Weighted F1"] /= repeat_times

    total_train_time /= repeat_times

    return results_sum, total_train_time, dimension


def main():
    base_folder = r'F:\datasheet\03时间序列分类\02urc\UCRArchive_2018'
    subfolders = [f for f in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, f))]
    subfolders = subfolders
    results_df = pd.DataFrame(columns=["Dataset"])
    
    cnt = 0

    for subfolder in subfolders:
        train_file = os.path.join(base_folder, subfolder, f"{subfolder}_TRAIN.tsv")
        test_file = os.path.join(base_folder, subfolder, f"{subfolder}_TEST.tsv")

        if os.path.exists(train_file) and os.path.exists(test_file):
            print(f"\n{cnt}-Processing dataset: {subfolder}")
            cnt += 1
            train_data = pd.read_csv(train_file, sep="\t", header=None)
            test_data = pd.read_csv(test_file, sep="\t", header=None)

            X_train = train_data.iloc[:, 1:].values
            y_train = train_data.iloc[:, 0].values
            X_test = test_data.iloc[:, 1:].values
            y_test = test_data.iloc[:, 0].values

            results, total_time, dimension = HITrocket_Liear_Pro(X_train, y_train, X_test, y_test, repeat_times=1)

            row = {"Dataset": subfolder}
            for model_name, metrics in results.items():
                row[f"{model_name} Train Time (s)"] = metrics["Train Time (s)"]
                row[f"{model_name} Weighted F1"] = metrics["Weighted F1"]
            row["Total Time (s)"] = total_time

            results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

    os.makedirs("./00linear_hitrocket_result", exist_ok=True)
    results_df.to_csv(f"./00linear_hitrocket_result/HITrocket_{dimension}_linear_results.csv", index=False)
    print(f"\n✅ Results saved to 'HITrocket_{dimension}_linear_results.csv'.")

    print("\n📊 Final Results:")
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()



0-Processing dataset: ACSF1
🔁 第 1/1 次训练
4400
    📐 模型: RidgeClassifierCV
    📐 模型: LogisticRegression


c:\Users\Einstein\anaconda3\envs\mycv\lib\site-packages\sklearn\linear_model\_ridge.py:2385: FutureWarning: 'store_cv_values' is deprecated in version 1.5 and will be removed in 1.7. Use 'store_cv_results' instead.
  warnings.warn(



1-Processing dataset: Adiac
🔁 第 1/1 次训练
4400
    📐 模型: RidgeClassifierCV
    📐 模型: LogisticRegression


c:\Users\Einstein\anaconda3\envs\mycv\lib\site-packages\sklearn\linear_model\_ridge.py:2385: FutureWarning: 'store_cv_values' is deprecated in version 1.5 and will be removed in 1.7. Use 'store_cv_results' instead.
  warnings.warn(



2-Processing dataset: AllGestureWiimoteX
🔁 第 1/1 次训练
4400
    📐 模型: RidgeClassifierCV
    📐 模型: LogisticRegression


...


126-Processing dataset: WormsTwoClass
🔁 第 1/1 次训练
4400
    📐 模型: RidgeClassifierCV
    📐 模型: LogisticRegression


c:\Users\Einstein\anaconda3\envs\mycv\lib\site-packages\sklearn\linear_model\_ridge.py:2385: FutureWarning: 'store_cv_values' is deprecated in version 1.5 and will be removed in 1.7. Use 'store_cv_results' instead.
  warnings.warn(



127-Processing dataset: Yoga
🔁 第 1/1 次训练
4400
    📐 模型: RidgeClassifierCV
    📐 模型: LogisticRegression


c:\Users\Einstein\anaconda3\envs\mycv\lib\site-packages\sklearn\linear_model\_ridge.py:2385: FutureWarning: 'store_cv_values' is deprecated in version 1.5 and will be removed in 1.7. Use 'store_cv_results' instead.
  warnings.warn(



✅ Results saved to 'HITrocket_4400_linear_results.csv'.

📊 Final Results:
                       Dataset  RidgeClassifierCV Train Time (s)  RidgeClassifierCV Weighted F1  LogisticRegression Train Time (s)  LogisticRegression Weighted F1  Total Time (s)
                         ACSF1                          0.010430                       0.868864                           0.335250                        0.929126        1.655887
                         Adiac                          0.060593                       0.793942                           8.732894                        0.818255       10.001381
            AllGestureWiimoteX                          0.041223                       0.591798                           2.438138                        0.638289        4.040881
            AllGestureWiimoteY                          0.040231                       0.590290                           2.644751                        0.665052        4.187896
            AllGestureWiimoteZ